In [21]:
import numpy as np
import pandas as pd

In [22]:
import os

path = "../../data/processed/"
dfs = {}

# read 03_CLEAN_COMPLETE_DF.parquet
sites = pd.read_parquet(os.path.join(path, "dep_codes.parquet"))


In [23]:


sites_names = sites[['HERlvl1Code', 'HERlvl1Name']]

# Mantener solo los valores únicos de 'col1'
sites_names = sites_names.drop_duplicates(subset='HERlvl1Code')

In [24]:
ranges = pd.read_csv('ranges.csv')
ranges

,HERlvl1Name,IBD_EQR_Status,IBD_min,IBD_max,IBD_mid,HERlvl1Code
0,ALPES INTERNES,Bad,0.000,9.800,9.30,2
1,ALPES INTERNES,Poor,9.800,13.225,10.30,2
2,ALPES INTERNES,Moderate,13.225,17.025,16.15,2
3,ALPES INTERNES,Good,17.025,18.725,17.90,2
4,ALPES INTERNES,High,18.725,20.000,19.55,2
...,...,...,...,...,...,...
105,VOSGES,Bad,0.000,9.550,7.90,4
106,VOSGES,Poor,9.550,12.775,11.20,4
107,VOSGES,Moderate,12.775,15.700,14.35,4
108,VOSGES,Good,15.700,18.075,17.05,4


In [ ]:
t = pd.read_csv('metricts_t.csv')
epm = pd.read_csv('metricts_epm.csv')
dirty = pd.read_csv('metricts_dirty.csv')

In [30]:
t

,region,R2_train,R2_valid,MAE_train,MAE_valid,RMSE_train,RMSE_valid,best_iterations
0,18,0.999984,0.901764,0.007065,0.475877,6.880693e-05,0.539792,2991
1,5,0.999617,0.929657,0.040646,0.425339,2.371315e-03,0.504880,2997
2,4,0.999998,0.833281,0.003072,0.743596,1.307719e-05,1.283097,2196
3,10,0.998872,0.949704,0.079430,0.381049,9.270532e-03,0.404585,2993
4,22,1.000000,0.562962,0.000040,1.190931,1.913874e-09,3.673909,1652
5,9,0.989169,0.941368,0.145762,0.265904,3.568258e-02,0.189518,2985
6,21,0.999565,0.928841,0.045767,0.463063,3.003642e-03,0.525956,2999
7,20,1.000000,0.867077,0.000260,0.726611,8.974746e-08,0.842552,2980
8,12,0.997816,0.944985,0.088146,0.353910,1.179671e-02,0.311098,2995
9,8,0.999931,0.842503,0.014798,0.613522,2.899673e-04,0.686089,1482


In [ ]:
import pandas as pd
from functools import reduce

# --- 1) gap = |R2_train - R2_valid| en cada df ---
def add_gap(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # asegúrate de que sean numéricos por si vienen como str
    out["R2_train"] = pd.to_numeric(out["R2_train"], errors="coerce")
    out["R2_valid"] = pd.to_numeric(out["R2_valid"], errors="coerce")
    out["gap"] = (out["R2_train"] - out["R2_valid"]).abs()
    return out

t_gap     = add_gap(t)
epm_gap   = add_gap(epm)
dirty_gap = add_gap(dirty)

# --- 2) renombrar todas las columnas excepto el id ('region') con sufijos t, epm, dirty ---
def with_suffix(df: pd.DataFrame, suf: str) -> pd.DataFrame:
    rename_map = {c: f"{c}_{suf}" for c in df.columns if c != "region"}
    return df.rename(columns=rename_map)

t_s     = with_suffix(t_gap, "t")
epm_s   = with_suffix(epm_gap, "epm")
dirty_s = with_suffix(dirty_gap, "dirty")

# --- 3) merge de los 3 por 'region' ---
merged = reduce(lambda l, r: pd.merge(l, r, on="region", how="outer"), [t_s, epm_s, dirty_s])

# --- 4) columnas resumen: quién tiene menor gap, mayor R2_train y mayor R2_valid ---
# (maneja empates concatenando con '+')
def winners(df: pd.DataFrame, cols_prefix: str, strip_prefix: str, how="min") -> pd.Series:
    cols = [c for c in df.columns if c.startswith(cols_prefix)]
    mat = df[cols]
    if how == "min":
        flags = mat.eq(mat.min(axis=1), axis=0)
    else:
        flags = mat.eq(mat.max(axis=1), axis=0)
    return flags.apply(lambda row: "+".join(row.index[row].str.replace(strip_prefix, "")), axis=1)

# menor gap
merged["min_gap_from"] = winners(merged, "gap_", "gap_")

# mayor R2_train
merged["max_R2_train_from"] = winners(merged, "R2_train_", "R2_train_", how="max")

# mayor R2_valid
merged["max_R2_valid_from"] = winners(merged, "R2_valid_", "R2_valid_", how="max")

# (opcional) reordenar para ver primero 'region' y los 3 resúmenes
summary_cols = ["region", "min_gap_from", "max_R2_train_from", "max_R2_valid_from"]
other_cols = [c for c in merged.columns if c not in summary_cols]
merged = merged[summary_cols + other_cols]




,region,min_gap_from,max_R2_train_from,max_R2_valid_from,R2_train_t,R2_valid_t,MAE_train_t,MAE_valid_t,RMSE_train_t,RMSE_valid_t,...,best_iterations_epm,gap_epm,R2_train_dirty,R2_valid_dirty,MAE_train_dirty,MAE_valid_dirty,RMSE_train_dirty,RMSE_valid_dirty,best_iterations_dirty,gap_dirty
0,1,t,dirty,t,0.999732,0.771942,0.019290,0.337553,0.000508,0.502332,...,2563,0.283926,0.999966,0.762488,0.006728,0.361597,6.388313e-05,0.523157,2999,0.237479
1,2,t,t,t,0.999985,0.700933,0.004353,0.196092,0.000025,0.279299,...,408,0.412661,0.999316,0.678723,0.025303,0.211337,1.154054e-03,0.300041,301,0.320593
2,3,t,dirty,t,0.996929,0.958966,0.122453,0.396599,0.022612,0.314263,...,2941,0.087254,0.998467,0.957690,0.086528,0.400418,1.129034e-02,0.324035,2999,0.040776
3,4,t,epm,t,0.999998,0.833281,0.003072,0.743596,0.000013,1.283097,...,2991,0.178084,1.000000,0.816799,0.000103,0.816917,1.472407e-08,1.409947,2641,0.183201
4,5,t,dirty,t,0.999617,0.929657,0.040646,0.425339,0.002371,0.504880,...,2930,0.096170,0.999879,0.927646,0.022638,0.448746,7.502270e-04,0.519312,2996,0.072233


In [32]:
merged

,region,min_gap_from,max_R2_train_from,max_R2_valid_from,R2_train_t,R2_valid_t,MAE_train_t,MAE_valid_t,RMSE_train_t,RMSE_valid_t,...,best_iterations_epm,gap_epm,R2_train_dirty,R2_valid_dirty,MAE_train_dirty,MAE_valid_dirty,RMSE_train_dirty,RMSE_valid_dirty,best_iterations_dirty,gap_dirty
0,1,t,dirty,t,0.999732,0.771942,0.019290,0.337553,5.078926e-04,0.502332,...,2563,0.283926,0.999966,0.762488,0.006728,0.361597,6.388313e-05,0.523157,2999,0.237479
1,2,t,t,t,0.999985,0.700933,0.004353,0.196092,2.528543e-05,0.279299,...,408,0.412661,0.999316,0.678723,0.025303,0.211337,1.154054e-03,0.300041,301,0.320593
2,3,t,dirty,t,0.996929,0.958966,0.122453,0.396599,2.261249e-02,0.314263,...,2941,0.087254,0.998467,0.957690,0.086528,0.400418,1.129034e-02,0.324035,2999,0.040776
3,4,t,epm,t,0.999998,0.833281,0.003072,0.743596,1.307719e-05,1.283097,...,2991,0.178084,1.000000,0.816799,0.000103,0.816917,1.472407e-08,1.409947,2641,0.183201
4,5,t,dirty,t,0.999617,0.929657,0.040646,0.425339,2.371315e-03,0.504880,...,2930,0.096170,0.999879,0.927646,0.022638,0.448746,7.502270e-04,0.519312,2996,0.072233
5,6,t,dirty,t,0.999606,0.945793,0.053201,0.502085,4.042070e-03,0.605457,...,2967,0.106782,0.999874,0.933357,0.029776,0.558042,1.287581e-03,0.744360,2951,0.066518
6,7,t,t,t,0.999996,0.910870,0.003824,0.261666,1.947808e-05,0.340529,...,472,0.138818,0.999899,0.878960,0.017750,0.284930,4.508090e-04,0.462445,769,0.120940
7,8,t,dirty,t,0.999931,0.842503,0.014798,0.613522,2.899673e-04,0.686089,...,2387,0.240440,1.000000,0.769914,0.000150,0.673607,3.010695e-08,1.002304,2906,0.230086
8,9,t,dirty,t,0.989169,0.941368,0.145762,0.265904,3.568258e-02,0.189518,...,2999,0.114165,0.992972,0.939320,0.119549,0.270092,2.315491e-02,0.196139,2996,0.053652
9,10,t,dirty,t,0.998872,0.949704,0.079430,0.381049,9.270532e-03,0.404585,...,2984,0.083650,0.999443,0.943318,0.055295,0.415377,4.575315e-03,0.455956,2999,0.056125


In [33]:
def choose_best_balanced(row, gap_threshold=0.15):
    # 1. modelo con mejor R²_valid
    best_r2 = row["max_R2_valid_from"]
    
    # 2. si su gap es alto, considerar al más estable
    gap_value = row[f"gap_{best_r2}"]
    if gap_value > gap_threshold:
        return row["min_gap_from"]  # cambio por estabilidad
    else:
        return best_r2

merged["best_final_model"] = merged.apply(choose_best_balanced, axis=1)

In [35]:
merged[["best_final_model",'region' ]]

,best_final_model,region
0,t,1
1,t,2
2,t,3
3,t,4
4,t,5
5,t,6
6,t,7
7,t,8
8,t,9
9,t,10


In [29]:
t_p = pd.read_csv('predicciones_t.csv')
epm_p = pd.read_csv('predicciones_epm.csv')
dirty_p = pd.read_csv('predicciones_dirty.csv')

In [ ]:
def pegargle_el_status